In [4]:
import ee
import geemap

PROJECT_ID = "bd-506013"

try:
    ee.Initialize(project=PROJECT_ID)
    print("Google Earth Engine initialized successfully.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print("Google Earth Engine initialized after authentication.")

# Area of interest definitions
admin_l1 = ee.FeatureCollection("FAO/GAUL/2015/level1")
admin_l2 = ee.FeatureCollection("FAO/GAUL/2015/level2")

gallura = admin_l2.filter(ee.Filter.eq("ADM2_NAME", "Sassari")).geometry().dissolve()
basilicata = admin_l1.filter(ee.Filter.eq("ADM1_NAME", "Basilicata")).geometry().dissolve()
apuane = admin_l2.filter(ee.Filter.inList("ADM2_NAME", ["Massa Carrara", "Lucca"])).geometry().dissolve()
trento = admin_l2.filter(ee.Filter.eq("ADM2_NAME", "Trento")).geometry().dissolve()

areas = {
    "Gallura": gallura,
    "Basilicata": basilicata,
    "Apuane": apuane,
    "Trento": trento,
}

# Landsat Collection 2 Level 2 surface reflectance settings
landsat_specs = [
    {
        "name": "Landsat 5 TM (1985-1991)",
        "collection_ids": ["LANDSAT/LT05/C02/T1_L2"],
        "start_date": "1985-06-01",
        "end_date": "1991-09-01",
    },
    {
        "name": "Landsat 7 ETM+ (2006-2010)",
        "collection_ids": ["LANDSAT/LE07/C02/T1_L2"],
        "start_date": "2006-06-01",
        "end_date": "2010-09-01",
    },
    {
        "name": "Landsat 8 OLI and Landsat 9 OLI-2 (2022-2025)",
        "collection_ids": ["LANDSAT/LC08/C02/T1_L2", "LANDSAT/LC09/C02/T1_L2"],
        "start_date": "2022-06-01",
        "end_date": "2025-09-01",
    },
]


def load_landsat_collection(collection_ids, geometry, start_date, end_date, cloud_threshold=20):
    collection = ee.ImageCollection(collection_ids[0])
    for collection_id in collection_ids[1:]:
        collection = collection.merge(ee.ImageCollection(collection_id))

    return (
        collection.filterBounds(geometry)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUD_COVER", cloud_threshold))
    )


landsat_collections = {
    area_name: {
        spec["name"]: load_landsat_collection(
            spec["collection_ids"],
            area_geometry,
            spec["start_date"],
            spec["end_date"],
        )
        for spec in landsat_specs
    }
    for area_name, area_geometry in areas.items()
}

# Interactive map
Map = geemap.Map(center=[42.5, 12.5], zoom=6)

styles = {
    "Gallura": {"color": "red", "fillColor": "00000000", "width": 2},
    "Basilicata": {"color": "blue", "fillColor": "00000000", "width": 2},
    "Apuane": {"color": "green", "fillColor": "00000000", "width": 2},
    "Trento": {"color": "purple", "fillColor": "00000000", "width": 2},
}

for area_name, area_geometry in areas.items():
    boundary = ee.FeatureCollection([ee.Feature(area_geometry)]).style(**styles[area_name])
    Map.addLayer(boundary, {}, f"AOI: {area_name}")

Map

Google Earth Engine initialized successfully.


Map(center=[42.5, 12.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [5]:
from IPython.display import display

# Apply the Landsat Collection 2 Level 2 reflectance scale factors.
def apply_landsat_scale(image):
    optical_bands = image.select("SR_B.").multiply(0.0000275).add(-0.2)
    thermal_bands = image.select("ST_B.*").multiply(0.00341802).add(149.0)
    return image.addBands(optical_bands, overwrite=True).addBands(thermal_bands, overwrite=True)


vis_params = {
    "Landsat 5 TM (1985-1991)": {"bands": ["SR_B3", "SR_B2", "SR_B1"], "min": 0.0, "max": 0.3, "gamma": 1.2},
    "Landsat 7 ETM+ (2006-2010)": {"bands": ["SR_B3", "SR_B2", "SR_B1"], "min": 0.0, "max": 0.3, "gamma": 1.2},
    "Landsat 8 OLI and Landsat 9 OLI-2 (2022-2025)": {"bands": ["SR_B4", "SR_B3", "SR_B2"], "min": 0.0, "max": 0.3, "gamma": 1.2},
}

for area_name, area_geometry in areas.items():
    area_map = geemap.Map()
    area_map.centerObject(area_geometry, 9)

    for spec in landsat_specs:
        collection = landsat_collections[area_name][spec["name"]]
        composite = apply_landsat_scale(collection.median()).clip(area_geometry)
        area_map.addLayer(composite, vis_params[spec["name"]], f"{area_name} - {spec['name']}")

    boundary = ee.FeatureCollection([ee.Feature(area_geometry)]).style(color="white", fillColor="00000000", width=2)
    area_map.addLayer(boundary, {}, f"{area_name} boundary")
    display(area_map)

Map(center=[40.749376559833, 8.949224103976439], controls=(WidgetControl(options=['position', 'transparent_bg'…

Map(center=[40.500331231968914, 16.082358221457387], controls=(WidgetControl(options=['position', 'transparent…

Map(center=[44.00622147440397, 10.438366462651464], controls=(WidgetControl(options=['position', 'transparent_…

Map(center=[46.135653672087976, 11.120957099510937], controls=(WidgetControl(options=['position', 'transparent…